# Multi-task Reward Router Training (Utility + Task Type)

This notebook trains a multi-task VLM router that predicts:
1. **Task Type**: Classification head (which router task is this?)
2. **Utility-based Reward**: Regression head (predicting utility for `accuracy`, `cheap`, `fast`, `balanced` modes).

It loads data from a canonical parquet file, builds a multi-task dataset, trains the `MultiTaskRewardRouterModel`, and provides inference helpers.

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import os
import sys
import logging
import json
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Union

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Set project root to allow importing modules if needed
# Assuming notebook is in artemis_final/router_train/notebooks
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent
sys.path.append(str(PROJECT_ROOT))

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("multitask_router")

print(f"Project root set to: {PROJECT_ROOT}")
print(f"Torch version: {torch.__version__}")

Project root set to: /storage/ice1/1/0/vchopra37/projects/vlm_router/artemis_final/router_train
Torch version: 2.9.1+cu128


In [14]:
class RouterConfig:
    # Data
    data_path = "/home/hice1/vchopra37/scratch/projects/vlm_router/artemis_final/router_train/notebooks/data/router_profiles_with_utility.parquet"
    
    # Model
    text_encoder_name = "distilbert-base-uncased"
    max_seq_len = 256
    model_emb_dim = 32
    mode_emb_dim = 16
    hidden_dim = 256
    dropout = 0.1
    
    num_workers = 4
    # Training
    batch_size = 512
    eval_batch_size=512
    epochs = 10
    lr_encoder = 2e-5
    lr_head = 3e-4
    weight_decay = 0.01
    lambda_task = 0.3
    gradient_clip = 1.0
    seed = 42
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

cfg = RouterConfig()

# Set seeds
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(f"Device: {cfg.device}")

Device: cuda


In [15]:
def load_and_prep_data(path):
    p = Path(path)
    if not p.exists():
        # Try searching for it
        p_alt = Path("/home/hice1/vchopra37/scratch/projects/vlm_router/artemis_final/router_train/notebooks/data/router_profiles_with_utility.parquet")
        if p_alt.exists():
            p = p_alt
        else:
            raise FileNotFoundError(f"Could not find data at {path} or {p_alt}")
    
    print(f"Loading data from {p}...")
    df = pd.read_parquet(p)
    
    # Filter valid rows
    # Required cols
    req_cols = ["sample_id", "router_task", "data_split", "prompt_text", "model_name", "ok"]
    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
    # Filter ok=True
    len_orig = len(df)
    df = df[df["ok"] == True].copy()
    print(f"Filtered ok=True: {len_orig} -> {len(df)}")
    
    return df

profiles_df = load_and_prep_data(cfg.data_path)
display(profiles_df.head(2))

Loading data from /home/hice1/vchopra37/scratch/projects/vlm_router/artemis_final/router_train/notebooks/data/router_profiles_with_utility.parquet...
Filtered ok=True: 339056 -> 339056


,sample_id,run_id,source_config,source_dataset,source_index,router_task,ground_truth_type,data_split,prompt_text,ground_truth,...,total_tokens,utility_accuracy,utility_cheap,utility_fast,utility_balanced,cost_norm_new,lat_norm,glider_score,judge_molmo_score,judge_molmo_rank_group
0,aokvqa_796_8644de03,run_20251207_225050,aokvqa,cauldron,796,knowledge_vqa,exact,test,How many women are holding umbrellas in front ...,Two.,...,493,0.0,0.986553,0.968584,0.488784,0.013447,0.031416,5.0,NaN,NaN
1,docvqa_270_ca57d063,run_20251207_225050,docvqa,cauldron,270,document_ocr,exact,train,What is the title of the document?\nEnsure bre...,Hazard and Exposure Criteria for Prioritizatio...,...,4876,0.0,0.866107,0.976502,0.460652,0.133893,0.023498,5.0,10.0,1.0


In [16]:
MODES = ["accuracy", "cheap", "fast", "balanced"]

def create_long_format(df, modes):
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building long-format"):
        base = {
            "sample_id": row["sample_id"],
            "router_task": row["router_task"],
            "data_split": row["data_split"],
            "prompt_text": row["prompt_text"],
            "model_name": row["model_name"],
            "prompt_len_words": row.get("txt_prompt_length_words", 0),
            "source_dataset": row.get("source_dataset", "unknown"),
        }
        
        # Add mode rows
        for mode in modes:
            # Check if utility_{mode} exists and is not null
            col_name = f"utility_{mode}"
            if col_name in row and pd.notnull(row[col_name]):
                r = base.copy()
                r["mode_name"] = mode
                r["utility_target"] = row[col_name]
                rows.append(r)
    
    return pd.DataFrame(rows)

df_long = create_long_format(profiles_df, MODES)
print(f"df_long shape: {df_long.shape}")
display(df_long.head())

Building long-format:   0%|          | 0/339056 [00:00<?, ?it/s]

df_long shape: (1356224, 9)


,sample_id,router_task,data_split,prompt_text,model_name,prompt_len_words,source_dataset,mode_name,utility_target
0,aokvqa_796_8644de03,knowledge_vqa,test,How many women are holding umbrellas in front ...,qwen2_5_vl_7b,29,cauldron,accuracy,0.000000
1,aokvqa_796_8644de03,knowledge_vqa,test,How many women are holding umbrellas in front ...,qwen2_5_vl_7b,29,cauldron,cheap,0.986553
2,aokvqa_796_8644de03,knowledge_vqa,test,How many women are holding umbrellas in front ...,qwen2_5_vl_7b,29,cauldron,fast,0.968584
3,aokvqa_796_8644de03,knowledge_vqa,test,How many women are holding umbrellas in front ...,qwen2_5_vl_7b,29,cauldron,balanced,0.488784
4,docvqa_270_ca57d063,document_ocr,train,What is the title of the document?\nEnsure bre...,qwen2_5_vl_7b,12,cauldron,accuracy,0.000000


In [17]:
# Create indices
model_names = sorted(df_long["model_name"].unique())
mode_names = MODES
task_names = sorted(df_long["router_task"].unique())

model_to_id = {m: i for i, m in enumerate(model_names)}
mode_to_id = {m: i for i, m in enumerate(mode_names)}
task_to_id = {t: i for i, t in enumerate(task_names)}

# Map to IDs
df_long["model_id"] = df_long["model_name"].map(model_to_id)
df_long["mode_id"] = df_long["mode_name"].map(mode_to_id)
df_long["task_id"] = df_long["router_task"].map(task_to_id)

# Save indices
os.makedirs("data", exist_ok=True)
with open("data/model_index.json", "w") as f: json.dump(model_names, f)
with open("data/mode_index.json", "w") as f: json.dump(mode_names, f)
with open("data/task_index.json", "w") as f: json.dump(task_names, f)

print("Indices saved to data/")
print(f"Models: {len(model_names)}")
print(f"Modes: {len(mode_names)}")
print(f"Tasks: {len(task_names)}")

Indices saved to data/
Models: 5
Modes: 4
Tasks: 30


In [18]:
# Compute class weights for task classification
# Helps stabilize task prediction when classes are imbalanced

task_counts = df_long["task_id"].value_counts().sort_index()
total = task_counts.sum()
class_weights = total / (len(task_counts) * task_counts)
class_weights = torch.tensor(class_weights.values, dtype=torch.float32, device=cfg.device)

print("Task counts:", task_counts.to_dict())
print(f"Class weights: {class_weights}")


Task counts: {0: 20052, 1: 59980, 2: 143144, 3: 40000, 4: 20000, 5: 40000, 6: 6000, 7: 40000, 8: 20000, 9: 118576, 10: 79960, 11: 67300, 12: 40000, 13: 39980, 14: 19980, 15: 40000, 16: 39968, 17: 25656, 18: 6260, 19: 40000, 20: 20000, 21: 39900, 22: 20000, 23: 86380, 24: 60000, 25: 143068, 26: 20000, 27: 20000, 28: 20000, 29: 20020}
Class weights: tensor([2.2545, 0.7537, 0.3158, 1.1302, 2.2604, 1.1302, 7.5346, 1.1302, 2.2604,
        0.3813, 0.5654, 0.6717, 1.1302, 1.1308, 2.2626, 1.1302, 1.1311, 1.7621,
        7.2216, 1.1302, 2.2604, 1.1330, 2.2604, 0.5234, 0.7535, 0.3160, 2.2604,
        2.2604, 2.2604, 2.2581], device='cuda:0')


In [19]:
from torch.utils.data import Dataset, DataLoader

class MultiTaskRewardRouterDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        """
        df: long-format DataFrame with columns:
            - prompt_text
            - source_dataset
            - prompt_len_words (or txt_prompt_length_words)
            - model_id
            - mode_id
            - task_id
            - sample_id
            - utility_target
        """
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ❌ IMPORTANT FIX: DO NOT INCLUDE router_task or data_split in the text.
        # We want the task head to infer router_task from content, not read the label.
        input_text = (
            f"[ROUTER] "
            f"Source: {row.get('source_dataset', 'unknown')}. "
            f"PromptLen: {row.get('prompt_len_words', 0)}. "
            f"Question: {row['prompt_text']}"
        )

        encoding = self.tokenizer(
            input_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "model_id": int(row["model_id"]),
            "mode_id": int(row["mode_id"]),
            "task_id": int(row["task_id"]),
            "utility_target": float(row["utility_target"]),
            "sample_id": row["sample_id"],
        }


def collate_fn(batch):
    """
    Pads input_ids / attention_mask and stacks all ids / labels.
    """
    input_ids = [b["input_ids"] for b in batch]
    attention_masks = [b["attention_mask"] for b in batch]

    # Pad sequences
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_masks_padded = torch.nn.utils.rnn.pad_sequence(
        attention_masks, batch_first=True, padding_value=0
    )

    model_ids = torch.tensor([b["model_id"] for b in batch], dtype=torch.long)
    mode_ids = torch.tensor([b["mode_id"] for b in batch], dtype=torch.long)
    task_ids = torch.tensor([b["task_id"] for b in batch], dtype=torch.long)
    utility_targets = torch.tensor(
        [b["utility_target"] for b in batch], dtype=torch.float
    )
    sample_ids = [b["sample_id"] for b in batch]

    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_masks_padded,
        "model_id": model_ids,
        "mode_id": mode_ids,
        "task_id": task_ids,
        "utility_target": utility_targets,
        "sample_ids": sample_ids,
    }



In [20]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.text_encoder_name)

# Split data
train_df = df_long[df_long["data_split"] == "train"]
val_df = df_long[df_long["data_split"] == "val"]
test_df = df_long[df_long["data_split"] == "test"]

In [21]:

# Instantiate datasets
max_len = getattr(cfg, "max_length", 256)

train_dataset = MultiTaskRewardRouterDataset(train_df, tokenizer, max_length=max_len)
val_dataset   = MultiTaskRewardRouterDataset(val_df,   tokenizer, max_length=max_len)
test_dataset  = MultiTaskRewardRouterDataset(test_df,  tokenizer, max_length=max_len)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
)

print(
    f"Datasets: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}"
)

Datasets: train=951912, val=204452, test=199860


In [22]:

# Verify batch
batch = next(iter(train_loader))
print("Batch shapes:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.shape}")

Batch shapes:
input_ids: torch.Size([512, 256])
attention_mask: torch.Size([512, 256])
model_id: torch.Size([512])
mode_id: torch.Size([512])
task_id: torch.Size([512])
utility_target: torch.Size([512])


In [23]:
import torch.nn as nn
from transformers import AutoModel

class MultiTaskRewardRouterModel(nn.Module):
    """
    Multi-head reward router:

    - Shared text encoder (e.g. DeBERTa).
    - Shared model embedding.
    - Shared mode embedding.
    - One regression head per mode:
        * accuracy
        * cheap
        * fast
        * balanced

    For each training row, we:
      - Look at its mode_id
      - Use ONLY that head to predict utility_hat

    Additionally:
      - A task classification head predicts router_task from text-only.
    """
    def __init__(self, config, num_models, num_modes, num_tasks, mode_names=None):
        super().__init__()
        self.config = config
        self.num_models = num_models
        self.num_modes = num_modes

        # Mode names should align with df_long / mode_to_id order
        if mode_names is None:
            mode_names = ["accuracy", "cheap", "fast", "balanced"]
        self.mode_names = mode_names

        # --- Text encoder ---
        self.text_encoder = AutoModel.from_pretrained(config.text_encoder_name)
        text_hidden_size = self.text_encoder.config.hidden_size

        # --- Embeddings ---
        self.model_embedding = nn.Embedding(num_models, config.model_emb_dim)
        self.mode_embedding  = nn.Embedding(num_modes,  config.mode_emb_dim)

        # --- Routing heads (one per mode) ---
        input_dim = text_hidden_size + config.model_emb_dim + config.mode_emb_dim

        def build_head():
            return nn.Sequential(
                nn.Linear(input_dim, config.hidden_dim),
                nn.ReLU(),
                nn.Dropout(config.dropout),
                nn.Linear(config.hidden_dim, config.hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(config.hidden_dim // 2, 1),
            )

        self.routing_heads = nn.ModuleList(
            [build_head() for _ in range(num_modes)]
        )

        # --- Task classification head (text-only) ---
        self.task_head = nn.Sequential(
            nn.Linear(text_hidden_size, config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, num_tasks),
        )

    def forward(self, input_ids, attention_mask, model_id, mode_id):
        """
        input_ids:      (B, L)
        attention_mask: (B, L)
        model_id:       (B,)
        mode_id:        (B,)  -- index in [0, num_modes)

        Returns:
            utility_hat: (B,)  -- scalar from the appropriate head per row
            task_logits: (B, num_tasks)
        """
        # --- Encode text ---
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # CLS representation
        h_text = outputs.last_hidden_state[:, 0, :]

        # --- Task prediction (text-only) ---
        task_logits = self.task_head(h_text)

        # --- Routing representation (text + model + mode) ---
        h_model = self.model_embedding(model_id)  # (B, model_emb_dim)
        h_mode  = self.mode_embedding(mode_id)    # (B, mode_emb_dim)

        h_combined = torch.cat([h_text, h_model, h_mode], dim=-1)  # (B, input_dim)

        # --- Compute all heads, then select per-row by mode_id ---
        # all_utilities: (B, num_modes)
        head_outputs = []
        for head in self.routing_heads:
            head_outputs.append(head(h_combined).squeeze(-1))
        all_utilities = torch.stack(head_outputs, dim=-1)  # (B, num_modes)

        # Gather correct utility for each row using mode_id
        utility_hat = all_utilities.gather(
            dim=1,
            index=mode_id.view(-1, 1)
        ).squeeze(1)  # (B,)

        return {
            "utility_hat": utility_hat,
            "task_logits": task_logits,
            # Optionally expose all_utilities if you want analysis:
            # "all_utilities": all_utilities,
        }


# ---- Instantiate the model ----

mode_names = MODES  # ["accuracy", "cheap", "fast", "balanced"]

model = MultiTaskRewardRouterModel(
    cfg,
    num_models=len(model_names),
    num_modes=len(mode_names),
    num_tasks=len(task_names),
    mode_names=mode_names,
)
model.to(cfg.device)

print("Model initialized:")
print(" - num_models:", len(model_names))
print(" - num_modes:", len(mode_names))
print(" - num_tasks:", len(task_names))


Model initialized:
 - num_models: 5
 - num_modes: 4
 - num_tasks: 30


In [26]:
# IMPORTANT: routing_mlp -> routing_heads (ModuleList)
num_training_steps = len(train_loader) * cfg.epochs
optimizer = torch.optim.AdamW(
    [
        {"params": model.text_encoder.parameters(),       "lr": cfg.lr_encoder},
        {"params": model.routing_heads.parameters(),      "lr": cfg.lr_head},
        {"params": model.task_head.parameters(),          "lr": cfg.lr_head},
        {"params": model.model_embedding.parameters(),    "lr": cfg.lr_head},
        {"params": model.mode_embedding.parameters(),     "lr": cfg.lr_head},
    ],
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_training_steps,
)

criterion_utility = nn.MSELoss()
criterion_task = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:

class RewardRouterTrainer:
    def __init__(
        self,
        model,
        optimizer,
        scheduler,
        train_loader,
        val_loader,
        criterion_utility,
        criterion_task,
        device,
        lambda_task=0.3,
        gradient_clip=1.0,
        epochs=10,
        patience=3,
        ckpt_path="best_multitask_router.pt",
    ):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion_utility = criterion_utility
        self.criterion_task = criterion_task
        self.device = device
        self.lambda_task = lambda_task
        self.gradient_clip = gradient_clip
        self.epochs = epochs
        self.patience = patience
        self.ckpt_path = ckpt_path

    def _to_device(self, batch):
        return {
            k: v.to(self.device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

    def _safe_corr(self, preds, targets):
        if len(preds) < 2:
            return 0.0
        try:
            corr = pearsonr(preds, targets)[0]
            if np.isnan(corr):
                return 0.0
            return float(corr)
        except Exception:
            return 0.0

    def train(self):
        history = []
        best_val_loss = float("inf")
        patience_counter = 0

        for epoch_idx in range(self.epochs):
            train_metrics = self.train_epoch()
            val_metrics = self.validate()

            epoch_log = {
                "epoch": epoch_idx + 1,
                **train_metrics,
                **val_metrics,
            }
            history.append(epoch_log)

            print(
                f"Epoch {epoch_idx + 1}: "
                f"train_loss={train_metrics['train_loss']:.4f} | "
                f"val_loss={val_metrics['val_loss']:.4f} | "
                f"train_task_acc={train_metrics['train_task_acc']:.4f} | "
                f"val_task_acc={val_metrics['val_task_acc']:.4f} | "
                f"train_corr={train_metrics['train_corr']:.4f} | "
                f"val_corr={val_metrics['val_corr']:.4f}"
            )

            if val_metrics["val_loss"] < best_val_loss:
                best_val_loss = val_metrics["val_loss"]
                torch.save(self.model.state_dict(), self.ckpt_path)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping triggered.")
                    break

        return history


class MultiTaskTrainer(RewardRouterTrainer):
    def train_epoch(self):
        self.model.train()
        total_loss = 0.0
        total_loss_routing = 0.0
        total_loss_task = 0.0
        task_correct = 0
        task_total = 0

        preds_u = []
        targets_u = []

        for batch in tqdm(self.train_loader, desc="Train epoch"):
            batch = self._to_device(batch)
            self.optimizer.zero_grad()

            out = self.model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                model_id=batch["model_id"],
                mode_id=batch["mode_id"],
            )

            loss_routing = self.criterion_utility(
                out["utility_hat"],
                batch["utility_target"],
            )
            loss_task = self.criterion_task(
                out["task_logits"],
                batch["task_id"],
            )
            loss = loss_routing + self.lambda_task * loss_task

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
            self.optimizer.step()
            if self.scheduler is not None:
                self.scheduler.step()

            total_loss += loss.item()
            total_loss_routing += loss_routing.item()
            total_loss_task += loss_task.item()

            preds = torch.argmax(out["task_logits"], dim=1)
            task_correct += (preds == batch["task_id"]).sum().item()
            task_total += batch["task_id"].numel()

            preds_u.extend(out["utility_hat"].detach().cpu().tolist())
            targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        avg_loss = total_loss / max(len(self.train_loader), 1)
        avg_loss_routing = total_loss_routing / max(len(self.train_loader), 1)
        avg_loss_task = total_loss_task / max(len(self.train_loader), 1)
        task_acc = task_correct / max(task_total, 1)
        train_corr = self._safe_corr(preds_u, targets_u)

        return {
            "train_loss": avg_loss,
            "train_loss_routing": avg_loss_routing,
            "train_loss_task": avg_loss_task,
            "train_task_acc": task_acc,
            "train_corr": train_corr,
        }

    def validate(self):
        self.model.eval()
        total_loss = 0.0
        total_loss_routing = 0.0
        total_loss_task = 0.0
        task_correct = 0
        task_total = 0

        preds_u = []
        targets_u = []

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Val epoch"):
                batch = self._to_device(batch)

                out = self.model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    model_id=batch["model_id"],
                    mode_id=batch["mode_id"],
                )

                loss_routing = self.criterion_utility(
                    out["utility_hat"],
                    batch["utility_target"],
                )
                loss_task = self.criterion_task(
                    out["task_logits"],
                    batch["task_id"],
                )
                loss = loss_routing + self.lambda_task * loss_task

                total_loss += loss.item()
                total_loss_routing += loss_routing.item()
                total_loss_task += loss_task.item()

                preds = torch.argmax(out["task_logits"], dim=1)
                task_correct += (preds == batch["task_id"]).sum().item()
                task_total += batch["task_id"].numel()

                preds_u.extend(out["utility_hat"].detach().cpu().tolist())
                targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        avg_loss = total_loss / max(len(self.val_loader), 1)
        avg_loss_routing = total_loss_routing / max(len(self.val_loader), 1)
        avg_loss_task = total_loss_task / max(len(self.val_loader), 1)
        task_acc = task_correct / max(task_total, 1)
        val_corr = self._safe_corr(preds_u, targets_u)

        return {
            "val_loss": avg_loss,
            "val_loss_routing": avg_loss_routing,
            "val_loss_task": avg_loss_task,
            "val_task_acc": task_acc,
            "val_corr": val_corr,
        }


trainer = MultiTaskTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_utility=criterion_utility,
    criterion_task=criterion_task,
    device=cfg.device,
    lambda_task=cfg.lambda_task,
    gradient_clip=cfg.gradient_clip,
    epochs=cfg.epochs,
    patience=3,
    ckpt_path="best_multitask_router.pt",
)

history = trainer.train()


Train epoch:   0%|          | 0/1860 [00:00<?, ?it/s]

Val epoch:   0%|          | 0/400 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>
Traceback (most recent call last):
Exception ignored in:   File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>    
self._shutdown_workers()Traceback (most recent call last):

<function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers

Traceback (most recent call last):
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/d

Epoch 1: train_loss=0.0505 | val_loss=0.0401 | train_task_acc=0.9698 | val_task_acc=0.9844 | train_corr=0.9273 | val_corr=0.9374


Train epoch:   0%|          | 0/1860 [00:00<?, ?it/s]

...

Val epoch:   0%|          | 0/400 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>
Traceback (most recent call last):
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/local/pace-apps/spack/packages/linux-rhel9-x86_64_v3/gcc-11.3.1/python-3.12.5-5sase6atfv2x5tf7dy5x5sqfzyguhsia/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^Exception ignored in: 
AssertionError: can only test a child process
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>
Traceback (most recent call last):
 

Epoch 3: train_loss=0.0166 | val_loss=0.0465 | train_task_acc=0.9994 | val_task_acc=0.9835 | train_corr=0.9484 | val_corr=0.9411


Train epoch:   0%|          | 0/1860 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>
Traceback (most recent call last):
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
     Exception ignored in:   ^^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ffe7907eca0>^
^Traceback (most recent call last):
^  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1654, in __del__
^^^    ^self._shutdown_workers()^^

  File "/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
  F

## 10.1 Training Curves


In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

# 1) Loss curves
plt.figure()
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# 2) Task accuracy
plt.figure()
plt.plot(history_df["epoch"], history_df["train_task_acc"], label="train_task_acc")
plt.plot(history_df["epoch"], history_df["val_task_acc"], label="val_task_acc")
plt.xlabel("Epoch")
plt.ylabel("Task Accuracy")
plt.title("Task Classification Accuracy")
plt.legend()
plt.show()

# 3) Correlation
plt.figure()
plt.plot(history_df["epoch"], history_df["train_corr"], label="train_corr")
plt.plot(history_df["epoch"], history_df["val_corr"], label="val_corr")
plt.xlabel("Epoch")
plt.ylabel("Pearson Corr (Utility)")
plt.title("Utility Correlation vs Epoch")
plt.legend()
plt.show()


## 11. Evaluation: Routing Accuracy vs Oracle


In [ ]:
ckpt_path = "best_multitask_router.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=cfg.device))
model.eval()
print(f"Loaded best model from {ckpt_path}")

# 11.1 Task classification accuracy on test set
test_task_correct = 0
test_task_total = 0

test_preds_u = []
test_targets_u = []
eval_records = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Eval"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"],
        )

        utility_hat = outputs["utility_hat"]
        task_logits = outputs["task_logits"]
        task_pred = task_logits.argmax(dim=-1)

        test_task_correct += (task_pred == batch["task_id"]).sum().item()
        test_task_total += batch["task_id"].numel()

        test_preds_u.extend(utility_hat.detach().cpu().tolist())
        test_targets_u.extend(batch["utility_target"].detach().cpu().tolist())

        for i in range(len(batch["sample_ids"])):
            eval_records.append({
                "sample_id": batch["sample_ids"][i],
                "mode_name": mode_names[int(batch["mode_id"][i].cpu())],
                "model_name": model_names[int(batch["model_id"][i].cpu())],
                "utility_target": float(batch["utility_target"][i].cpu()),
                "utility_pred": float(utility_hat[i].cpu()),
                "task_id": int(batch["task_id"][i].cpu()),
                "task_name": task_names[int(batch["task_id"][i].cpu())],
                "task_pred_id": int(task_pred[i].cpu()),
                "task_pred_name": task_names[int(task_pred[i].cpu())],
            })

test_task_acc = test_task_correct / max(test_task_total, 1)
try:
    test_corr = float(pearsonr(test_preds_u, test_targets_u)[0])
except Exception:
    test_corr = 0.0

print(f"Test task classification accuracy: {test_task_acc:.4f}")
print(f"Test utility correlation: {test_corr:.4f}")

eval_df = pd.DataFrame(eval_records)
display(eval_df.head())


In [ ]:
print("Evaluating Routing Accuracy vs Oracle on test set...")
routing_metrics_per_mode = []

for mode in mode_names:
    mode_df = eval_df[eval_df["mode_name"] == mode]

    total_samples = 0
    routing_hits = 0
    oracle_utility_sum = 0.0
    router_utility_sum = 0.0

    for sid, group in mode_df.groupby("sample_id"):
        if len(group) < 2:
            continue

        oracle_row = group.loc[group["utility_target"].idxmax()]
        pred_row = group.loc[group["utility_pred"].idxmax()]

        total_samples += 1
        if pred_row["model_name"] == oracle_row["model_name"]:
            routing_hits += 1

        oracle_utility_sum += oracle_row["utility_target"]
        router_utility_sum += pred_row["utility_target"]

    if total_samples > 0:
        oracle_mean = oracle_utility_sum / total_samples
        router_mean = router_utility_sum / total_samples
        gap = oracle_mean - router_mean
        recovery = router_mean / oracle_mean if oracle_mean > 0 else 0.0

        routing_metrics_per_mode.append({
            "mode": mode,
            "routing_accuracy": routing_hits / total_samples,
            "oracle_utility_mean": oracle_mean,
            "router_utility_mean": router_mean,
            "utility_gap": gap,
            "recovery": recovery,
            "test_task_acc": test_task_acc,
            "test_utility_corr": test_corr,
        })

summary = pd.DataFrame(routing_metrics_per_mode)
display(summary)

if not summary.empty:
    ax = summary.set_index("mode")[
        ["routing_accuracy", "recovery"]
    ].plot(kind="bar", figsize=(8, 4), ylim=(0, 1.05), title="Routing Accuracy and Recovery (Test)")
    ax.set_ylabel("Score")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


## 12. Save Artifacts


In [ ]:
os.makedirs("results", exist_ok=True)

if 'eval_df' in globals():
    eval_df.to_csv("results/multitask_eval_details.csv", index=False)
    print("Saved detailed eval to results/multitask_eval_details.csv")
if 'summary' in globals():
    summary.to_csv("results/multitask_eval_summary.csv", index=False)
    print("Saved summary to results/multitask_eval_summary.csv")


## 13. Inference Demo: Routing + Task Prediction


In [ ]:
id_to_task = {v: k for k, v in task_to_id.items()}

def route_query(
    model,
    tokenizer,
    prompt_text: str,
    mode_name: str,
    model_names: List[str],
    model_to_id: Dict[str, int],
    mode_to_id: Dict[str, int],
    id_to_task: Dict[int, str],
    device: str = "cuda",
    temperature: float = 1.0,
    task_top_k: int = 3,
):
    model.eval()

    input_text = (
        f"[ROUTER] Task: unknown. "
        f"Source: inference. "
        f"PromptLen: {len(prompt_text.split())}. "
        f"Question: {prompt_text}"
    )

    encoding = tokenizer(
        input_text,
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    num_cands = len(model_names)
    input_ids = input_ids.repeat(num_cands, 1)
    attention_mask = attention_mask.repeat(num_cands, 1)

    model_ids = torch.tensor([model_to_id[m] for m in model_names], device=device)
    mode_id_val = mode_to_id.get(mode_name, 0)
    mode_ids = torch.tensor([mode_id_val] * num_cands, device=device)

    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            model_id=model_ids,
            mode_id=mode_ids,
        )

        utility_scores = out["utility_hat"]
        task_logits = out["task_logits"][0]

    # Model routing
    scores = utility_scores / temperature
    probs = torch.softmax(scores, dim=0)

    scores_dict = {m: float(s) for m, s in zip(model_names, utility_scores)}
    probs_dict = {m: float(p) for m, p in zip(model_names, probs)}

    # Task prediction
    task_probs = torch.softmax(task_logits, dim=0)
    task_pred_idx = int(task_probs.argmax().item())
    task_pred_name = id_to_task[task_pred_idx]
    task_conf = float(task_probs[task_pred_idx])

    k = min(task_top_k, len(task_probs))
    topk_probs, topk_indices = torch.topk(task_probs, k=k)
    task_topk = [
        {"task": id_to_task[int(idx)], "prob": float(p)}
        for p, idx in zip(topk_probs, topk_indices)
    ]

    task_probs_dict = {
        id_to_task[i]: float(task_probs[i])
        for i in range(len(task_probs))
    }

    return {
        "scores": scores_dict,
        "probs": probs_dict,
        "task_probs": task_probs_dict,
        "task_type_pred": task_pred_name,
        "task_confidence": task_conf,
        "task_topk": task_topk,
    }


In [ ]:
from pprint import pprint

sid = test_df["sample_id"].sample(1, random_state=cfg.seed).iloc[0]
sample = test_df[test_df["sample_id"] == sid].iloc[0]
prompt_text = sample["prompt_text"]

mode_name = "accuracy"
model_candidates = list(model_to_id.keys())

out = route_query(
    model=model,
    tokenizer=tokenizer,
    prompt_text=prompt_text,
    mode_name=mode_name,
    model_names=model_candidates,
    model_to_id=model_to_id,
    mode_to_id=mode_to_id,
    id_to_task=id_to_task,
    device=cfg.device,
    temperature=1.0,
    task_top_k=3,
)

print("PROMPT:", prompt_text[:500], "...")
print("Predicted task:", out["task_type_pred"], f"(conf={out['task_confidence']:.2f})")
print("Top-3 tasks:")
pprint(out["task_topk"])
print("Model scores:")
pprint(out["scores"])
print("Model probs:")
pprint(out["probs"])
